# BRISC — DRL Local (TD3-ready)

**Runs against the local repo `D:\\iteris` and the local `fyp_env` (GPU).** No Drive, no clone.

### Run in VSCode
1. Install the **Python** + **Jupyter** extensions.
2. Open this file -> top-right **Select Kernel** -> **Python Environments** -> `fyp_env`
   (`F:\\Anaconda\\envs\\fyp_env\\python.exe`).
3. Edit the **EDIT ME** paths in section 0, then **Run All** (or run cell-by-cell).

| Picker | Options |
|---|---|
| `TUMOR_TYPE` | 'all' | 'glioma' | 'meningioma' | 'pituitary' |
| `AGENT_NAME` | `'DuelingDDQN'` (discrete contour) · `'TD3'` (continuous contour) |

**First run:** leave the dry-run (section 3) ON to validate the whole path in ~2-3 min before the full run.

## 0 · Local setup  — EDIT THE PATHS BELOW

In [ ]:
import sys, os
from pathlib import Path
os.environ['PYTHONIOENCODING'] = 'utf-8'   # Windows-safe unicode in prints

# ================= EDIT ME =================
PKG_ROOT       = Path(r'D:\iteris')                 # local repo (has iteris/ + configs/)
DATA_ROOT      = r'D:\iteris\_local_data\brisc2025'   # dataset root
BASELINE_CKPT  = r'D:\iteris\_local_data\brisc_best.pt'   # U-Net baseline checkpoint you downloaded
CKPT_DIR       = Path(r'D:\iteris\_local_ckpt')    # DRL checkpoints + figures land here
# ==========================================

CKPT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PKG_ROOT))

assert (PKG_ROOT / 'iteris' / '__init__.py').exists(), f'iteris package not found under {PKG_ROOT}'
assert Path(DATA_ROOT).exists(), f'DATA_ROOT does not exist: {DATA_ROOT}'
assert Path(BASELINE_CKPT).exists(), f'baseline checkpoint not found: {BASELINE_CKPT} — download it first'

import torch
print('Package :', PKG_ROOT)
print('Data    :', DATA_ROOT)
print('Ckpt    :', BASELINE_CKPT)
print('GPU     :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (training will be slow)')

## 1 · Configure — pick target + algorithm

In [ ]:
# ==============================================================================
TUMOR_TYPE = 'all'        # 'all' | 'glioma' | 'meningioma' | 'pituitary'
AGENT_NAME = 'DuelingDDQN'   # 'DuelingDDQN' = discrete contour refinement | 'TD3' = continuous contour refinement
# ==============================================================================

_AGENTS = ('DuelingDDQN', 'TD3')
assert AGENT_NAME in _AGENTS, f'AGENT_NAME must be one of {_AGENTS}'
_CONFIG_MAP = {'all': 'brisc_drl_tumor.yaml', 'glioma': 'brisc_drl_glioma.yaml',
               'meningioma': 'brisc_drl_meningioma.yaml', 'pituitary': 'brisc_drl_pituitary.yaml'}
assert TUMOR_TYPE in _CONFIG_MAP, f"TUMOR_TYPE must be one of {list(_CONFIG_MAP)}"

from iteris.config import load_drl_class_config, resolve_agent_config, load_config

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / _CONFIG_MAP[TUMOR_TYPE]))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)
baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

# Inject local paths (override the Kaggle defaults baked into the YAML)
baseline_cfg['data_root']  = str(DATA_ROOT)
cfg['baseline_checkpoint'] = str(BASELINE_CKPT)
cfg['checkpoint_dir']      = str(CKPT_DIR)

print(f"agent={cfg['agent_type']}  env={cfg.get('env_class','default')}  "
      f"reward={cfg.get('reward_mode')}  class={cfg.get('class_name', cfg.get('target_class'))}")

## 2 · Warm-start — pre-compute U-Net init masks

Runs the U-Net once over all splits and caches `{image, gt_mask, init_mask, prob_map}`.
The DRL loop never touches the U-Net again.

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.005),
    tumor_type_filter   = cfg.get('tumor_type_filter', None),
)

print(f'\nSamples - train {len(train_samples)}  val {len(val_samples)}  test {len(test_samples)}')
init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6)) for s in val_samples]
print(f'U-Net init Dice on val: mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · Dry-run (recommended on first launch) — ~2-3 min

Trains 600 steps on a small slice to confirm warm-start -> agent -> env -> reward all work,
without mutating `cfg` / samples. Set `RUN_DRY_RUN = False` once it passes.

In [ ]:
RUN_DRY_RUN = True   # set False to skip once validated

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training
    _dry = copy.deepcopy(cfg)
    _dry.update(dict(train_steps=600, prefill_steps=400, buffer_size=3000,
                     eval_every=300, epsilon_decay_steps=400, batch_size=32))
    _r = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f"\nDry run OK. (best score at 600 steps is not meaningful: {_r['best_dice']:.4f})")
    print(f"  cfg['train_steps'] still = {cfg['train_steps']} (unchanged)")
else:
    print('Dry run skipped.')

## 4 · Full training

**4 GB GPU note:** if you hit `CUDA out of memory`, uncomment the `batch_size` line below.
Contour-env agents (TD3 / DuelingDDQN) are CPU-bound on the spline rasteriser, so a
full local run can take hours — local is best for dry-runs/debugging; do full runs on Kaggle/Colab.

In [ ]:
from iteris.drl_training import run_drl_training

# cfg['batch_size'] = 32      # <- uncomment on a 4 GB GPU if you OOM
result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']
best_dice = result['best_dice']
print(f"\nDone - {cfg['agent_type']} | best val final-Dice {best_dice:.4f}")
print(f"  checkpoint: {result['checkpoint']}")

## 5 · Visualisation setup

In [ ]:
from iteris.refinement_viz import (refinement_env_kwargs, refinement_env_cls,
    build_replays, plot_comparison, plot_playback, plot_behaviour, evaluate_testset)

ENV_KW     = refinement_env_kwargs(cfg)
ENV_CLS    = refinement_env_cls(cfg)   # SegmentationEnv, or ContourRefineEnv for TD3/contour
replays    = build_replays(agent, val_samples, ENV_KW, n_viz=8, seed=0, env_cls=ENV_CLS)
CLASS_NAME = cfg.get('class_name', '')
print(f'Built {len(replays)} greedy replays.')

## 6 · Sample comparison — best / median / worst gains

In [ ]:
import matplotlib.pyplot as plt
fig = plot_comparison(replays, baseline_cfg, cfg,
    class_idx=cfg.get('target_class', 1), class_name=CLASS_NAME,
    out_path=str(CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_comparison.png"))
plt.show()

## 7 · Iteration playback — all steps on best-gain sample

In [ ]:
fig = plot_playback(replays[-1], cfg, class_name=CLASS_NAME,
    out_path=str(CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_playback.png"))
plt.show()

## 8 · Behaviour analysis — trajectories + action distribution

In [ ]:
fig = plot_behaviour(replays, cfg, class_name=CLASS_NAME,
    out_path=str(CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_behaviour.png"))
plt.show()

## 9 · Learning curves

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['step'], history['init_dice_mean'], '--', color='gray', alpha=0.6, label='Init (U-Net)')
ax1.plot(history['step'], history['final_dice_mean'], color='C0', label=f"Final ({cfg['agent_type']})")
if 'best_dice_mean' in history.columns:
    ax1.plot(history['step'], history['best_dice_mean'], color='C2', alpha=0.7, label='Best-seen')
ax1.set(xlabel='step', ylabel='Val Dice', title=f"{cfg['dataset']} {CLASS_NAME} refinement"); ax1.legend(); ax1.grid(alpha=0.3)
delta = history['final_dice_mean'] - history['init_dice_mean']
ax2.plot(history['step'], delta, color='C0'); ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(history['step'], delta, 0, where=(delta > 0), alpha=0.15, color='green')
ax2.fill_between(history['step'], delta, 0, where=(delta < 0), alpha=0.15, color='red')
ax2.set(xlabel='step', ylabel='delta Dice (final - init)', title='Refinement gain'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_curves.png"), dpi=150)
plt.show()

## 10 · Test-set evaluation + summary JSON

In [ ]:
import json
test_metrics = evaluate_testset(agent, test_samples, ENV_KW, env_cls=ENV_CLS)
print(json.dumps(test_metrics, indent=2))
summary = {**test_metrics, 'agent_type': cfg['agent_type'], 'target_class': cfg['target_class'],
           'class_name': cfg.get('class_name', ''), 'dataset': cfg['dataset'],
           'tumor_type': TUMOR_TYPE}
out = str(CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_test_results.json")
with open(out, 'w') as f: json.dump(summary, f, indent=2)
print('\nSaved', out)
print(f"Baseline (U-Net) Dice: {test_metrics['init_dice_mean']:.4f}")
print(f"Refined  (agent)  Dice: {test_metrics['final_dice_mean']:.4f}  (delta {test_metrics['delta_dice_mean']:+.4f})")